In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Detecting Covariate Shift\n",
    "\n",
    "**The problem**: Your model was trained on data from Distribution A. In production, you're seeing data from Distribution B. The model doesn't know the difference—it just makes predictions. But those predictions might be garbage.\n",
    "\n",
    "**This notebook**: We'll simulate this scenario and use three different methods to detect when our input distribution has shifted."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "# Our drift detection tools\n",
    "import sys\n",
    "sys.path.append('..')\n",
    "from drift_detection import KSTest, PSI, MMD\n",
    "\n",
    "sns.set_style(\"whitegrid\")\n",
    "plt.rcParams['figure.figsize'] = [10, 6]\n",
    "\n",
    "np.random.seed(42)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## The Scenario\n",
    "\n",
    "Imagine we've deployed a breast cancer screening AI. It was trained on mammograms from Hospital A—an urban academic medical center with newer digital equipment and a diverse patient population.\n",
    "\n",
    "Now it's being used at Hospital B—a rural community hospital with older equipment and a different demographic mix (older patients on average).\n",
    "\n",
    "We'll simulate this with two features:\n",
    "- **Pixel intensity** (affected by scanner calibration)\n",
    "- **Patient age** (different demographics)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Reference distribution: Hospital A (training data)\n",
    "n_reference = 2000\n",
    "\n",
    "ref_intensity = np.random.normal(loc=0.50, scale=0.12, size=n_reference)\n",
    "ref_age = np.random.normal(loc=52, scale=10, size=n_reference)\n",
    "\n",
    "reference_data = np.column_stack([ref_intensity, ref_age])\n",
    "reference_df = pd.DataFrame(reference_data, columns=['pixel_intensity', 'age'])\n",
    "reference_df['source'] = 'Reference (Hospital A)'\n",
    "\n",
    "print(f\"Reference data: {len(reference_df)} samples\")\n",
    "reference_df.describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Current distribution: Hospital B (production data)\n",
    "n_current = 1500\n",
    "\n",
    "# Different scanner -> different pixel intensity distribution\n",
    "cur_intensity = np.random.normal(loc=0.55, scale=0.14, size=n_current)\n",
    "\n",
    "# Older patient population\n",
    "cur_age = np.random.normal(loc=58, scale=12, size=n_current)\n",
    "\n",
    "current_data = np.column_stack([cur_intensity, cur_age])\n",
    "current_df = pd.DataFrame(current_data, columns=['pixel_intensity', 'age'])\n",
    "current_df['source'] = 'Current (Hospital B)'\n",
    "\n",
    "print(f\"Current data: {len(current_df)} samples\")\n",
    "current_df.describe()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Visualizing the Shift\n",
    "\n",
    "Let's see what this drift looks like visually."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "combined_df = pd.concat([reference_df, current_df])\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "\n",
    "sns.kdeplot(data=combined_df, x='pixel_intensity', hue='source', \n",
    "            fill=True, alpha=0.3, ax=axes[0])\n",
    "axes[0].set_title('Pixel Intensity Distribution', fontsize=14)\n",
    "axes[0].set_xlabel('Normalized Pixel Intensity')\n",
    "\n",
    "sns.kdeplot(data=combined_df, x='age', hue='source', \n",
    "            fill=True, alpha=0.3, ax=axes[1])\n",
    "axes[1].set_title('Patient Age Distribution', fontsize=14)\n",
    "axes[1].set_xlabel('Age (years)')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "print(\"Visual inspection: Both features appear shifted. But is it statistically significant?\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Method 1: Kolmogorov-Smirnov Test\n",
    "\n",
    "The KS test compares the cumulative distribution functions of two samples.\n",
    "\n",
    "**Pros**: Simple, interpretable, no hyperparameters  \n",
    "**Cons**: Only works on one feature at a time"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ks_detector = KSTest(alpha=0.05)\n",
    "\n",
    "print(\"KS Test Results\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "for feature in ['pixel_intensity', 'age']:\n",
    "    result = ks_detector.detect(\n",
    "        reference_df[feature].values,\n",
    "        current_df[feature].values\n",
    "    )\n",
    "    print(f\"\\n{feature}:\")\n",
    "    print(f\"  KS statistic: {result.statistic:.4f}\")\n",
    "    print(f\"  p-value: {result.p_value:.2e}\")\n",
    "    print(f\"  Drift detected: {'YES' if result.drift_detected else 'NO'}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Method 2: Population Stability Index (PSI)\n",
    "\n",
    "PSI is the industry standard in banking and insurance.\n",
    "\n",
    "**Interpretation**:  \n",
    "- PSI < 0.1: No significant shift  \n",
    "- PSI 0.1 - 0.2: Moderate shift, investigate  \n",
    "- PSI > 0.2: Significant shift, take action"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "psi_calculator = PSI(n_bins=10, binning='quantile')\n",
    "\n",
    "print(\"PSI Results\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "for feature in ['pixel_intensity', 'age']:\n",
    "    result = psi_calculator.calculate(\n",
    "        reference_df[feature].values,\n",
    "        current_df[feature].values\n",
    "    )\n",
    "    print(f\"\\n{feature}:\")\n",
    "    print(f\"  PSI: {result.psi:.4f}\")\n",
    "    print(f\"  Status: {result.drift_level.upper()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Method 3: Maximum Mean Discrepancy (MMD)\n",
    "\n",
    "MMD compares the full joint distribution, not just individual features.\n",
    "\n",
    "**Pros**: Works on high-dimensional data, catches multivariate shifts  \n",
    "**Cons**: Slower, less interpretable"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "mmd_detector = MMD()\n",
    "\n",
    "print(\"MMD Test Results (Joint Distribution)\")\n",
    "print(\"=\" * 50)\n",
    "\n",
    "result = mmd_detector.detect(\n",
    "    reference_data,\n",
    "    current_data,\n",
    "    permutation_test=True,\n",
    "    n_permutations=100\n",
    ")\n",
    "\n",
    "print(f\"\\nMMD: {result.mmd:.4f}\")\n",
    "print(f\"p-value: {result.p_value:.4f}\")\n",
    "print(f\"Drift detected: {'YES' if result.drift_detected else 'NO'}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Summary\n",
    "\n",
    "All three methods detected drift. In a real deployment, this should trigger:\n",
    "\n",
    "1. **Alert** the ML ops team\n",
    "2. **Investigate** the root cause\n",
    "3. **Decide** whether to recalibrate, retrain, or halt"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\" * 60)\n",
    "print(\" DRIFT DETECTION SUMMARY\")\n",
    "print(\"=\" * 60)\n",
    "print(\"\\nAll methods detected significant drift.\")\n",
    "print(\"Recommended action: Investigate root cause before continuing deployment.\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}